In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import re
import struct

from scipy.interpolate import interp1d
from pathlib import Path

In [2]:
def cargar_senal(path_csv):
    df = pd.read_csv(path_csv, sep=';')
    df.columns = [col.strip() for col in df.columns]
    return df

In [3]:
def upsample_rom(df):
    # Obtener el número total de filas (filas de datos)
    original_len = len(df)
    
    # Contar las celdas no vacías en la columna de ángulos (ROM)
    no_vacias_angulos = df.iloc[:, 1].notna().sum()
    
    # Si hay valores válidos, se puede hacer upsampling
    if no_vacias_angulos == 0:
        raise ValueError("No hay valores válidos en la columna de ángulos.")
    
    # Calcular el factor de upsampling
    factor_upsampling = original_len / no_vacias_angulos

    #ROM Sampling Rate = 20 Hz

    sampling_rate = factor_upsampling * 20
    
    # Crear el espacio de interpolación para ROM
    x_original = np.linspace(0, 1, no_vacias_angulos)
    x_new = np.linspace(0, 1, original_len)
    
    # Crear DataFrame vacío para los datos upsampled
    upsampled_df = df.copy()  # Copiar el DataFrame original para no modificar las otras columnas
    
    # Interpolación solo de la columna de ROM
    #rom_col = 'angles_biceps_brachii_(right)_triceps_brachii_long_(right)'
    interpolator = interp1d(x_original, df.iloc[:, 1].dropna(), kind='cubic', fill_value='extrapolate')
    upsampled_df.iloc[:, 1] = interpolator(x_new)
    
    return upsampled_df, sampling_rate

In [4]:
def calcular_aceleracion(df, sampling_rate):
    #rom_col = 'angles_biceps_brachii_(right)_triceps_brachii_long_(right)'
    tiempo = np.arange(len(df)) / sampling_rate
    
    d_speed = np.diff(df.iloc[:, 1]) / np.diff(tiempo)
    d_speed = np.insert(d_speed, 0, d_speed[0])
    
    d_acceleration = np.diff(d_speed) / np.diff(tiempo)
    d_acceleration = np.insert(d_acceleration, 0, d_acceleration[0])
    
    d_jerk = np.diff(d_acceleration) / np.diff(tiempo)
    d_jerk = np.insert(d_jerk, 0, d_jerk[0])

    #Save in the dataframe only coherent values (fix first values of acceleration and jerk)
    
    d_acceleration[0] = d_acceleration[2]
    d_acceleration[1] = d_acceleration[2]

    d_jerk[0] = d_jerk[3]
    d_jerk[1] = d_jerk[3]
    d_jerk[2] = d_jerk[3]
    
    df['speed'] = d_speed
    df['acceleration'] = d_acceleration
    df['jerk'] = d_jerk
    
    return df

In [5]:
def resolver_offsets_emg(df):
    for col in df.columns:
        if 'emg_muscle' in col:
            offset = df[col].mean()
            df[col] = df[col] - offset
    return df

In [6]:
def normalizar_tiempo(df):
    n = len(df)
    df['tiempo_normalizado'] = np.linspace(0, 1, n)
    return df

In [7]:
def columnas_identificadoras(df, path):
    carpetas = os.path.dirname(path).split('/')
    #print(carpetas)
    del carpetas[0]
    #print(carpetas)
    df['class'] = [carpetas[0]] * len(df)
    bpm = re.findall(r'\d+', carpetas[1])
    df['BPM'] = [int(bpm[0])] * len(df)
    df['kid number'] = [carpetas[2]] * len(df)
    return df

In [8]:
def borrar_y_renombrar(df):
    df = df.drop(['angles_window', 'tiempo_normalizado'], axis = 1)
    df.columns = ['ROM', 'Biceps EMG', 'Triceps EMG', 'Speed', 'Acceleration', 'Jerk', 'Class', 'BPM', 'Kid number']
    return df

In [9]:
def save_to_htk(df, filename, samp_period=100000, parm_kind=9):
    """
    Guarda un DataFrame como archivo HTK binario.

    Args:
        df (pd.DataFrame): DataFrame con solo las columnas de características (float).
        filename (str): Ruta al archivo .htk.
        samp_period (int): Tiempo entre vectores, en unidades de 100 ns (100000 = 10 ms).
        parm_kind (int): Código del tipo de datos (9 = USER).
    """
    # Asegúrate de que solo hay datos numéricos (sin strings)
    features = df.select_dtypes(include=[np.number]).values.astype(np.float32)
    nSamples, vector_size = features.shape

    sampSize = vector_size * 4  # cada float32 ocupa 4 bytes

    with open(filename, 'wb') as f:
        # Escribir header
        f.write(struct.pack('>i', nSamples))         # número de vectores
        f.write(struct.pack('>i', samp_period))      # periodo de muestreo
        f.write(struct.pack('>h', sampSize))         # bytes por vector
        f.write(struct.pack('>h', parm_kind))        # tipo de parámetros

        # Escribir datos
        for vector in features:
            f.write(struct.pack('>' + 'f' * vector_size, *vector))

In [10]:
htk_files = []
lab_files = []

In [11]:
root = Path('MEDICIONES_COLEGIO')
for csv_file in root.rglob('advanced_*.csv'):
    if '.ipynb_checkpoints' in csv_file.parts:
        continue
    path = csv_file.as_posix()
    #print(path)
    df = cargar_senal(path)
    df_copy = df.copy()
    df, sampling_rate = upsample_rom(df)
    df = resolver_offsets_emg(df)
    df = calcular_aceleracion(df, sampling_rate)
    df = normalizar_tiempo(df)
    df = columnas_identificadoras(df, path)
    df = borrar_y_renombrar(df)
    feature_cols = ['ROM', 'Biceps EMG', 'Triceps EMG', 'Speed', 'Acceleration', 'Jerk']
    df_features = df[feature_cols]
    htk_filename = f"htk_files/data/{df['Class'].iloc[0]}_{df['BPM'].iloc[0]}_{df['Kid number'].iloc[0]}.htk"
    lab_filename = f"htk_files/data/{df['Class'].iloc[0]}_{df['BPM'].iloc[0]}_{df['Kid number'].iloc[0]}.lab"
    save_to_htk(df_features, htk_filename)
    htk_files.append(htk_filename)
    with open(lab_filename, 'w', encoding='utf-8') as f:
        f.write(df['Class'].iloc[0])
    lab_files.append(lab_filename)
    df = None

In [12]:
excel_dir = Path("training")
train_excels = sorted(excel_dir.glob("Train*.xlsx"))

for excel_file in train_excels:
    train_df = pd.read_excel(excel_file, sheet_name=0)
    scp_data = []
    j = 0
    while j < len(train_df):
        scp_data.append(f"data/{train_df['Class'].iloc[j]}_{train_df['BPM'].iloc[j]}_{train_df['Kid number'].iloc[j]}.htk")
        j+=1
    #scp_filename = f"{excel_file[10:5]}.scp" #training\\TrainXX.xlsx -> TrainXX
    scp_filename = f"htk_files/{excel_file.stem}.scp"
    with open(scp_filename, 'w', encoding='utf-8') as f:
        for scp in scp_data:
            f.write(f"{scp}\n")

In [4]:
#Para crear de 0 lab_files
lab_path = "htk_files/data"
lab_files = []
for archivo in os.listdir(lab_path):
    ruta_completa = os.path.join(lab_path, archivo)
    if os.path.isfile(ruta_completa) and archivo.endswith('.lab'):
        lab_files.append(ruta_completa)

In [5]:
mlf_path = "htk_files/labels.mlf"
with open(mlf_path, 'w', encoding='utf-8') as f:
    f.write("#!MLF!#\n")
    for file in lab_files:
        filepath = file[15:]
        f.write(f"\"*/{filepath}\"\n")
        with open(file, 'r', encoding='utf-8') as f2:
            f.write(f"{f2.readline()}\n.\n")